# Time Series Analysis: Anomaly Detection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import meteostat as ms
from datetime import datetime

In [ ]:
bolzano = ms.Point(46.5, 11.35, 262)  

# Get nearby weather stations
stations = ms.stations.nearby(bolzano, limit=5, radius=1000000)
station_ids = stations.index.tolist()

start = datetime(2023, 1, 1)
end = datetime(2023, 2, 28)

ms.config.block_large_requests = False
data = ms.hourly(station=station_ids, start=start, end=end, parameters=["temp"])

df_temp = data.fetch().reset_index().pivot(index='time', columns='station', values='temp')
df_temp

## Matrix Profile

### Naive Implementation

In [ ]:
from scipy.stats import zscore

def matrix_profile_naive(time_series, window_size):
    n = len(time_series)
    n_windows = n - window_size + 1
    exclusion_zone = window_size // 4

    matrix_profile = np.full(n_windows, np.inf)
    matrix_profile_index = np.zeros(n_windows, dtype=int)

    # Loop through Distance Matrix
    for i in range(n_windows):
        window_i = zscore(time_series[i:i+window_size])

        for j in range(n_windows):
            if abs(i - j) <= exclusion_zone:
                continue

            window_j = zscore(time_series[j:j+window_size])

            dist = np.linalg.norm(window_i - window_j)

            if dist < matrix_profile[i]:
                matrix_profile[i] = dist
                matrix_profile_index[i] = j
    
    return matrix_profile, matrix_profile_index

### STOMP Implementation

#### Helper Functions

In [ ]:
from scipy.fft import fft, ifft

def compute_mean_std(time_series, window_size):
    X_padded = np.insert(time_series, 0, 0)

    sum_X = np.cumsum(X_padded)
    sum_X_sq = np.cumsum(X_padded ** 2)

    rolling_sum_X = sum_X[window_size:] - sum_X[:-window_size]
    mean = rolling_sum_X / window_size

    rolling_sum_X_sq = sum_X_sq[window_size:] - sum_X_sq[:-window_size]
    std_sq = (rolling_sum_X_sq / window_size) - mean**2
    std = np.sqrt(std_sq)

    return mean, std

def sliding_dot_product(query, time_series):
    n = len(time_series)
    m = len(query)

    padded_time_series = np.append(time_series, np.zeros(n))
    reversed_query = np.flip(query)
    padded_reversed_query = np.append(reversed_query, np.zeros(2*n - m))

    fft_query = fft(padded_reversed_query)
    fft_time_series = fft(padded_time_series)

    dot_products = ifft(np.multiply(fft_query, fft_time_series)).real
    return dot_products[m-1:n]

def calculate_distance_profile(dot_products, means, stds, idx, window_size):
    query_mean = means[idx]
    query_std = stds[idx]

    centered_dot_product = dot_products - (window_size * means * query_mean)
    norm_factor = (window_size * stds * query_std)
    
    pearson_correlation = centered_dot_product / norm_factor

    distances = np.sqrt(2 * window_size * (1 - pearson_correlation))
    return distances

def element_wise_min(profile, profile_index, candidate, idx):
    update_min_mask = candidate < profile

    profile[update_min_mask] = candidate[update_min_mask]
    profile_index[update_min_mask] = idx

    return profile, profile_index

In [ ]:
def matrix_profile_stomp(time_series, window_size):
    n = len(time_series)
    n_windows = n - window_size + 1
    exclusion_zone = int(np.ceil(window_size / 4))

    mean, std = compute_mean_std(time_series, window_size)
    dot = sliding_dot_product(time_series[:window_size], time_series)
    dot_first = dot.copy()

    dist = calculate_distance_profile(dot_first, mean, std, 0, window_size)
    dist[0:min(n_windows, exclusion_zone + 1)] = np.inf

    matrix_profile = dist.copy()
    matrix_profile_index = np.ones(n_windows)

    for i in range(1, n_windows):
        # Apply speedup trick
        for j in range(n_windows-1, 0, -1):
            dot[j] = (dot[j-1] 
                      - time_series[j-1] * time_series[i-1]
                      + time_series[j+window_size-1] * time_series[i+window_size-1]
            )
        dot[0] = dot_first[i]
        
        dist = calculate_distance_profile(dot, mean, std, i, window_size)

        exclusion_start = max(0, i - exclusion_zone)
        exclusion_end = min(n_windows, i + exclusion_zone + 1)
        dist[exclusion_start : exclusion_end] = np.inf
        
        matrix_profile, matrix_profile_index = element_wise_min(
            matrix_profile, matrix_profile_index, dist, i
        )
        
    return matrix_profile, matrix_profile_index

### Experiments

In [ ]:
ts = df_temp["16014"].to_numpy()

mn, mni = matrix_profile_naive(ts, 20)

In [ ]:
plt.plot(mn)

In [ ]:
mps, mpsi = matrix_profile_stomp(ts, 20)

In [ ]:
print(f"Naive and STOMP are equal: {np.allclose(mn, mps)}")

plt.plot(mps)
plt.show()

## Arima

## DWT-MLEAD

In [ ]:
from scipy.stats import chi2
from collections import deque

class DWT_MLEAD():
    def __init__(self, max_levels, level_base, level_order, forgetting_factor, significance_level, global_anomaly_threshold):
        # 1: Parameter definition
        self.max_levels = max_levels
        self.level_base = level_base
        self.level_order = level_order
        self.forgetting_factor = forgetting_factor
        self.significance_level = significance_level
        self.global_anomaly_threshold = global_anomaly_threshold

        # 2: Variable initialization
        self.window_sizes = [max(1, int(np.floor(level_base ** (level_order - level)))) 
                             for level in range(max_levels)]
        self.global_event_counter = 0.0
        self.discount_factor = ((self.window_sizes[-1] - 1) / (self.window_sizes[-1] + 1))
        self.allow_anomaly = True

        # Holding weight sum, rolling mean, inverse scatter matrix, scatter matrix
        self.approx_dist_state = [(0, np.zeros(dim), np.identity(dim), np.identity(dim)) for dim in self.window_sizes]
        self.detail_dist_state = [(0, np.zeros(dim), np.identity(dim), np.identity(dim)) for dim in self.window_sizes]

        # 3: Helping variables
        self.approx_windows = [deque(maxlen=dim) for dim in self.window_sizes]
        self.detail_windows = [deque(maxlen=dim) for dim in self.window_sizes]
        
        self.prev_approx = [0.0] * self.max_levels

    def dwt_mlead(self, time_step, current_value):
        """time_step expects 1-based indexing"""

        # 1: Determine the maximum active frequency level
        max_active_level = 0
        for level in range(self.max_levels - 1, -1, -1):
            if time_step % (2 ** level) == 0:
                max_active_level = level
                break
        
        predictions_sum = 0
        current_approx = current_value

        for level in range(max_active_level + 1):
            # Online Haare discrete wavelet transform
            if level == 0:
                approx_coeff = current_approx
                detail_coeff = current_approx
            else:
                prev_approx = self.prev_approx[level - 1]
                
                approx_coeff = (prev_approx + current_approx) / np.sqrt(2.0)
                detail_coeff = (prev_approx - current_approx) / np.sqrt(2.0)

            if level == max_active_level:
                self.prev_approx[level] = approx_coeff

            current_approx = approx_coeff

            # Update sliding windows
            self.approx_windows[level].append(approx_coeff)
            self.detail_windows[level].append(detail_coeff)

            if len(self.approx_windows[level]) < self.window_sizes[level]:
                continue
            
            # Update distributions
            detail_window = np.array(self.detail_windows[level])
            self.detail_dist_state[level] = self.update(self.detail_dist_state[level], detail_window, self.forgetting_factor)

            prediction = self.predict(self.detail_dist_state[level], detail_window, self.significance_level) 

            if level > 0:
                # Level 0 contains no approximation coefficients
                approx_window = np.array(self.approx_windows[level])
                self.approx_dist_state[level] = self.update(self.approx_dist_state[level], approx_window, self.forgetting_factor)

                prediction += self.predict(self.approx_dist_state[level], approx_window, self.significance_level) 

            # Accumulate predictions
            predictions_sum += prediction

        # Adjust global event counter
        self.global_event_counter = (
            self.discount_factor * self.global_event_counter + predictions_sum
        )

        # Flag anomaly
        anomaly_occured = (self.allow_anomaly 
                           and self.global_event_counter >= self.global_anomaly_threshold)

        if anomaly_occured:
            self.allow_anomaly = False

        # Allow new anomaly
        elif self.global_event_counter < 2.0/3.0 * self.global_anomaly_threshold:
            self.allow_anomaly = True

        return anomaly_occured

    def update(self, dist_state, window, forgetting_factor):
        weight_sum, rolling_mean, inv_scatter_matrix, scatter_matrix = dist_state

        weight_sum = forgetting_factor * weight_sum + 1

        delta_window_mean = window - rolling_mean
        rolling_mean = rolling_mean + (1.0/weight_sum * delta_window_mean)
        delta_window_mean_updated = window - rolling_mean

        scatter_matrix = forgetting_factor * scatter_matrix + np.outer(delta_window_mean, delta_window_mean_updated) #  matrix is optional
        
        # Using Sherman-Morrison formula
        inv_scatter_matrix = (
            1.0/forgetting_factor * inv_scatter_matrix 
            - (1.0/forgetting_factor * np.outer(inv_scatter_matrix @ delta_window_mean, delta_window_mean_updated @ inv_scatter_matrix)
               / (forgetting_factor + np.dot(delta_window_mean_updated, inv_scatter_matrix @ delta_window_mean))
            )
        )

        return (weight_sum, rolling_mean, inv_scatter_matrix, scatter_matrix)

    def predict(self, dist_state, window, significance_level):
        weight_sum, rolling_mean, inv_scatter_matrix, scatter_matrix = dist_state

        delta_window_mean = window - rolling_mean
        mahalanobis_distance = weight_sum * np.dot(delta_window_mean, inv_scatter_matrix @ delta_window_mean)

        dim = len(window)
        signifance_threshold = chi2.ppf(1.0 - significance_level, df=dim)

        # Return event flag
        return 1.0 if mahalanobis_distance > signifance_threshold else 0.0

## Exercises

Compare the runtime of the manual matrix profile implementation to the one by tslearn.

Parallelize STOMP

Make the matrix inversion in DWT-MLEAD floating-error safe.

1. Use np.linalg.inv to always recalculate the inverse scatter matrix.
2. Compare the algorithms over a long time series. Do the results differ? Does the execution time differ?
3. Think of a way to prevent this error without loosing too much efficiency (e.g. periodic recalculation, regularization term etc.). Compare the refined approach with the other implementations.